# S6_5 — Pareto Frontier & Label Efficiency Deep Dive

**Leaf-JEPA IRP** | Stage 6 — Analysis and Interpretation


**Purpose:** Produce the flagship figures for RQ2 (Pareto frontier), RQ3 (PEFT vs full FT gap),
and RQ5 (label efficiency crossover with supervised CNNs).

**Outputs:**
- `pareto_frontier_params_vs_f1.png` — RQ2 headline figure
- `label_efficiency_all_methods.png` — RQ3/RQ5 headline figure
- `aulec_comparison.csv` — area under label efficiency curve per method
- `crossover_analysis.json` — fraction where PEFT surpasses supervised
- `radar_method_profiles.png` — multi-metric comparison (optional)

**Research Questions Served:** RQ2, RQ3, RQ5


## Initialization

In [1]:
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage6_analysis_and_interpretation.config_stage6 import *
from stage6_analysis_and_interpretation.analysis_utils import (
    set_seed, load_json, save_json,
    compute_pareto_frontier, plot_pareto,
    compute_aulec, find_crossover,
    plot_label_efficiency, plot_radar,
    print_section,
)

set_seed(RANDOM_SEED)
STAGE6_FIGURES.mkdir(parents=True, exist_ok=True)
STAGE6_TABLES.mkdir(parents=True, exist_ok=True)
STAGE6_DATA.mkdir(parents=True, exist_ok=True)


## Pareto results

In [ ]:
# Collect all method results for Pareto
print_section("Collecting Results for Pareto Plot")

# Build list of (trainable_params, macro_f1, method_name)
pareto_points = []

# Baselines
baseline_params = {
    "B1": 25_600_000, "B2": 12_200_000,
    "B3": 0, "B4": 632_000_000, "B5": 0,
}
# B3 and B5 have only linear head params — estimate
linear_head_params = EMBED_DIM * NUM_CLASSES  # 1280 * 38 = 48,640
baseline_params["B3"] = linear_head_params
baseline_params["B5"] = linear_head_params

for bid in BASELINE_IDS:
    agg_path = BASELINES_DIR / f"{bid}_aggregate.json"
    if agg_path.exists():
        agg = load_json(agg_path)
        f1 = agg.get("macro_f1_mean", agg.get("macro_f1", 0))
        if isinstance(f1, list):
            f1 = np.mean(f1)
        params = baseline_params.get(bid, 0)
        pareto_points.append((params, float(f1), BASELINE_NAMES.get(bid, bid)))
        print(f"  {bid} ({BASELINE_NAMES.get(bid, bid)}): params={params:,}, F1={f1:.4f}")

# PEFT methods
s1_path = PEFT_RESULTS_DIR / "S1_method_comparison_summary.json"
if s1_path.exists():
    s1 = load_json(s1_path)
    if isinstance(s1, dict):
        for method_name, method_data in s1.items():
            if isinstance(method_data, dict):
                f1 = method_data.get("macro_f1_mean", method_data.get("macro_f1", 0))
                if isinstance(f1, list):
                    f1 = np.mean(f1)
                params = method_data.get("trainable_params", method_data.get("params", 100000))
                pareto_points.append((params, float(f1), method_name))
                print(f"  {method_name}: params={params:,}, F1={f1:.4f}")

print(f"\n  Total points: {len(pareto_points)}")


## Pareto Frontier plot

In [ ]:
print_section("Pareto Frontier")

if len(pareto_points) >= 2:
    pareto_optimal = compute_pareto_frontier(pareto_points)
    print(f"  Pareto-optimal methods: {[p[2] for p in pareto_optimal]}")

    # Find full FT F1 for RQ3 threshold
    full_ft_f1 = None
    for p, f1, name in pareto_points:
        if "Full FT" in name or "B4" in name or name == "I-JEPA Full FT":
            full_ft_f1 = f1
            break

    plot_pareto(pareto_points, pareto_optimal,
                STAGE6_FIGURES / "pareto_frontier_params_vs_f1.png",
                method_colours=METHOD_COLOURS,
                rq3_threshold=RQ3_THRESHOLD_PCT,
                full_ft_f1=full_ft_f1)

    # Save pareto data
    save_json([{"params": p, "f1": f1, "name": name, "pareto_optimal": (p, f1, name) in pareto_optimal}
               for p, f1, name in pareto_points],
              STAGE6_DATA / "pareto_points.json")

    # RQ3 check
    if full_ft_f1:
        threshold = full_ft_f1 - RQ3_THRESHOLD_PCT / 100.0
        within_threshold = [(p, f1, n) for p, f1, n in pareto_points
                           if f1 >= threshold and p < 632_000_000 * 0.02]
        print(f"\n  RQ3: Methods within {RQ3_THRESHOLD_PCT}% of full FT ({full_ft_f1:.4f}) "
              f"AND < 2% params:")
        for p, f1, n in within_threshold:
            pct = p / 632_000_000 * 100
            gap = (full_ft_f1 - f1) * 100
            print(f"    {n}: F1={f1:.4f} ({gap:.2f}pp below FT), {pct:.2f}% params")
else:
    print("  \u26a0\ufe0f Need at least 2 data points for Pareto")


## Label efficiency overlay

In [ ]:
print_section("Label Efficiency Analysis")

le_data = {}

# Load baseline label efficiency
for bid in ["B1", "B2"]:
    le_path = BASELINES_DIR / f"{bid}_label_efficiency.json"
    if le_path.exists():
        le = load_json(le_path)
        if isinstance(le, dict) and "fractions" in le:
            le_data[BASELINE_NAMES.get(bid, bid)] = le
            print(f"  {bid}: loaded label efficiency data")

# Load PEFT label efficiency
s2_path = PEFT_RESULTS_DIR / "S2_label_efficiency_results.json"
if s2_path.exists():
    s2 = load_json(s2_path)
    if isinstance(s2, dict):
        for method_name, method_le in s2.items():
            if isinstance(method_le, dict) and "fractions" in method_le:
                le_data[method_name] = method_le
                print(f"  {method_name}: loaded label efficiency data")

if le_data:
    plot_label_efficiency(le_data, STAGE6_FIGURES / "label_efficiency_all_methods.png",
                          method_colours=METHOD_COLOURS)

    # AULEC
    print("\n  AULEC (Area Under Label Efficiency Curve):")
    aulec_results = {}
    for name, data in le_data.items():
        aulec = compute_aulec(data["fractions"], data["mean"])
        aulec_results[name] = aulec
        print(f"    {name}: {aulec:.4f}")
    pd.DataFrame([aulec_results]).T.rename(columns={0: "AULEC"}).to_csv(
        STAGE6_TABLES / "aulec_comparison.csv")

    # Crossover analysis
    print("\n  Crossover Analysis (PEFT vs Supervised):")
    crossover_results = {}
    supervised_methods = ["ResNet-50", "EfficientNet-B3"]
    for sup_name in supervised_methods:
        if sup_name not in le_data:
            continue
        for peft_name in le_data:
            if peft_name in supervised_methods:
                continue
            cross = find_crossover(
                le_data[sup_name]["fractions"],
                le_data[sup_name]["mean"],
                le_data[peft_name]["mean"]
            )
            if cross is not None:
                crossover_results[f"{peft_name}_vs_{sup_name}"] = cross
                print(f"    {peft_name} surpasses {sup_name} at {cross:.1%} labels")
            else:
                print(f"    {peft_name} vs {sup_name}: no crossover found")

    save_json(crossover_results, STAGE6_DATA / "crossover_analysis.json")
else:
    print("  \u26a0\ufe0f No label efficiency data found")


## Radae chart

In [ ]:
print_section("Radar Chart (Multi-Metric Profiles)")

# Build method profiles with normalised metrics
# You'll need to fill these from your actual results
# Example structure:
metric_names = ["Macro-F1", "Param Efficiency", "Train Speed", "Inference Speed", "VRAM Efficiency", "Cross-Domain F1"]

method_profiles = {}

# Try to build profiles from available data
for name in list(le_data.keys()) + ["Linear Probe"]:
    # This is a template — fill with actual values from your experiments
    pass

# If we have at least 2 methods with full profiles, plot
if len(method_profiles) >= 2:
    plot_radar(method_profiles, metric_names,
               STAGE6_FIGURES / "radar_method_profiles.png",
               method_colours=METHOD_COLOURS)
else:
    print("  \u26a0\ufe0f Radar chart requires manual data entry.")
    print("  Fill method_profiles dict with normalised [0,1] values for each metric.")
    print("  Example:")
    print('    method_profiles = {')
    print('        "LoRA r=8": [0.95, 0.98, 0.85, 0.90, 0.92, 0.78],')
    print('        "Full FT":  [0.98, 0.01, 0.40, 0.90, 0.30, 0.80],')
    print('    }')

print("\n\u2705 S6_5 complete.")


## Critical Analysis

### Pareto Frontier Interpretation
- Methods ON the frontier are Pareto-optimal: no other method achieves higher F1 with fewer params
- The frontier SHAPE reveals the accuracy-efficiency tradeoff — a steep early section means
  small parameter budgets give large F1 gains; a flat section means diminishing returns
- Methods BELOW the frontier are strictly dominated and should be acknowledged as such

### Label Efficiency Crossover
- The crossover point is the headline finding for RQ5
- If PEFT crosses over supervised CNNs at ≤10% labels, the research hypothesis is confirmed
- If no crossover exists, discuss why (dataset too simple, supervised features transfer well)

### AULEC
- Higher AULEC = better across ALL label fractions
- Use AULEC in Chapter 5 to rank methods when the curves cross each other multiple times
